# Tratamento de dados dengue

Notebook para carregar os parquets de 2017, 2018 e 2019, aplicar a classe de limpeza e exibir o dataframe tratado.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

for candidate in (Path.cwd(), Path.cwd() / "notebooks_limpeza"):
    if (candidate / "dengue_data_cleaner.py").exists():
        sys.path.insert(0, str(candidate))
        break

from dengue_data_cleaner import DengueDataCleaner

In [ ]:
cleaner = DengueDataCleaner()

df_angel = cleaner.limpar_angel()
df_pedro = cleaner.limpar_pedro()
df_ruan = cleaner.limpar_ruan()
df_tratado = cleaner.juntar(df_angel, df_pedro, df_ruan)
df = df_tratado

print("Original:", cleaner.df.shape)
print("Angel:", df_angel.shape)
print("Pedro:", df_pedro.shape)
print("Ruan:", df_ruan.shape)
print("Tratado:", df_tratado.shape)

In [ ]:
df_tratado['final_classification'].value_counts()

In [ ]:
print(list(df_tratado.columns))
display(df_tratado)

Tratamento  para ML

In [ ]:
#One Hot Encoding para sexo
df_tratado = pd.get_dummies(df_tratado, columns=["sex_label"], dtype=int)
df_tratado = df_tratado.drop(columns=["sex"], errors="ignore")
df_tratado = df_tratado.rename(columns = {'sex_label_Feminino':'sex_Female', 'sex_label_Ignorado':'sex_Ignored', 'sex_label_Masculino': 'sex_Male'}).copy()


#Analise ciclica de meses do ano e semanas epidemiológicas
#Aqui, temos os meses representados num circulo --> (cos, sen)
df_tratado['notification_month_sin'] = np.sin(2 * np.pi * df_tratado['notification_month'] / 12)
df_tratado['notification_month_cos'] = np.cos(2 * np.pi * df_tratado['notification_month'] / 12)


df_tratado["symptom_epi_week_number"].max() 
#Max é 53
df_tratado['symptom_epi_week_number_sin'] = np.sin(2 * np.pi * df_tratado['symptom_epi_week_number'] / 53)
df_tratado['symptom_epi_week_number_cos'] = np.cos(2 * np.pi * df_tratado['symptom_epi_week_number'] / 53)

#Criar uma coluna para quantidade total de sintomas.
sintomas = [
    'fever', 'myalgia', 'headache', 'rash', 'vomiting', 'nausea',
    'back_pain', 'conjunctivitis', 'arthritis', 'joint_pain',
    'petechiae', 'retro_orbital_pain'
]

#Ja verifiquei: 1 para positivo e 2 para falso. Cada um tem 58 NaN

df_tratado[sintomas] = df_tratado[sintomas].replace(2,0)

#for x in sintomas: 
#   print(x)
#   print(df_tratado[x].value_counts(dropna=False))
#   print()

#Agora 1 = Tem ; 0 = Não tem
df_tratado['number_of_symptoms'] = df_tratado[sintomas].sum(axis = 1)

#Rash e Retro Orbital Pain são os sintomas que apresentam maior gap relativo entre descartados x confirmados(Como falado na apresentação)

sintomas_importantes = ['rash', 'retro_orbital_pain']

df_tratado['number_of_important_symptoms'] = df_tratado[sintomas_importantes].sum(axis = 1)

df_tratado['important_symptoms'] = (df_tratado['number_of_important_symptoms'] > 0).astype(int)

df_tratado['both_important_symptoms'] = ((df_tratado['rash'] == 1) & (df_tratado['retro_orbital_pain'] == 1)).astype(int)

df_tratado['education_level_label'].value_counts(dropna = False)

#Ordinal Encoding na escolaridade
map_escolaridade = {
    'Ignorado': 0,
    None: 0, 
    'Não se aplica': 0,

    'Analfabeto': 1,

    '1ª a 4ª série incompleta': 2,
    '4ª série completa': 2,
    '5ª à 8ª série incompleta': 2,

    'Ensino fundamental completo': 3,

    'Ensino médio incompleto': 4,
    'Ensino médio completo': 4,

    'Educação superior incompleta': 5,
    'Educação superior completa': 5

}

df_tratado['education_level'] = df_tratado['education_level_label'].map(map_escolaridade)

df_tratado = df_tratado.drop(columns = ['disease_code']) #Disease_code todos são A90.

print(df_tratado['days_to_notification'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999])) #Aqui vemos que valores acima de 60 fazem parte de 1% dos dados

df_tratado = df_tratado[(df_tratado['days_to_notification'] <= 60) | (df_tratado['days_to_notification'].isna())].copy() #Mantemos os 99% que são coerentes
df_tratado['days_to_notification'].value_counts()

#Gravidez agora é binario, 1 para sim e 0 para não. Alem disso, temos agora uma coluna binaria para dizer se a gravidez foi informada(exclui o caso dos homens e ignorado)
df_tratado['pregnancy'] = df_tratado['pregnancy_status'].isin([1, 2, 3, 4]).astype(int) 

df_tratado['pregnancy_informed'] = df_tratado['pregnancy_status'].isin([1, 2, 3, 4, 5]).astype(int)

df_tratado = df_tratado.drop(columns = ['pregnancy_status_label', 'pregnancy_status'], errors='ignore').copy()

df_tratado["occupation_code"] = df_tratado["occupation_code"].astype("string") #parquet tava reclamando do tipo de occupation code

cleaner.salvar_df(df_tratado, "dados_tratados/dengue_tratado_ml.parquet")



Analisando a quantidade de linhas que temos para cada ano. Observamos que 2017 e 2018 têm bem menos linhas que 2019. Sendo assim, faremos uma divisão mais justa na hora de separar treino e teste.

In [ ]:
df_tratado['notification_year'].value_counts()

#df_teste = df_tratado[df_tratado['notification_year'] == 2019]

#df_teste['notification_month'].value_counts()